# 01 — Génération du dataset synthétique
### CHU Mohammed VI — Agadir | Module Hospitalisations

**Objectif de ce notebook :**
- Générer des données synthétiques réalistes avec `Faker` : patients, services, hospitalisations
- Injecter volontairement des anomalies (valeurs manquantes, doublons, dates incohérentes, âges invalides)
- Exporter les fichiers CSV bruts dans `data/raw/`

Ces fichiers "sales" serviront de point de départ au notebook `02_audit_qualite_donnees.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta
import os

# Reproductibilité : toujours les mêmes données générées à chaque exécution
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('fr_FR')
Faker.seed(SEED)

print("Librairies chargées avec succès ✅")

Librairies chargées avec succès ✅


## 1. Paramètres de configuration

⚠️ Commence avec de petits volumes pour valider que tout fonctionne, puis augmente `N_PATIENTS` et `N_HOSPITALISATIONS` une fois validé (ex. 8000 / 15000 pour le volume complet du cahier des charges).

In [2]:
# --- Volumes (mode test : petit volume pour valider rapidement) ---
N_PATIENTS = 1000
N_HOSPITALISATIONS = 1000

# --- Chemin de sortie ---
OUTPUT_DIR = "../data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Villes marocaines réalistes ---
VILLES_MAROC = [
    "Agadir", "Casablanca", "Marrakech", "Rabat", "Tanger",
    "Fès", "Meknès", "Oujda", "Kénitra", "Tétouan",
    "Safi", "Taroudant", "Essaouira", "Inezgane", "Chtouka Aït Baha"
]

# --- Services hospitaliers (8 à 10 comme demandé au cahier des charges) ---
SERVICES = [
    {"nom_service": "Cardiologie", "categorie": "Médecine", "capacite_lits": 30},
    {"nom_service": "Pédiatrie", "categorie": "Médecine", "capacite_lits": 25},
    {"nom_service": "Chirurgie générale", "categorie": "Chirurgie", "capacite_lits": 35},
    {"nom_service": "Urgences", "categorie": "Urgences", "capacite_lits": 20},
    {"nom_service": "Maternité", "categorie": "Gynéco-Obstétrique", "capacite_lits": 40},
    {"nom_service": "Oncologie", "categorie": "Médecine", "capacite_lits": 20},
    {"nom_service": "Néphrologie", "categorie": "Médecine", "capacite_lits": 15},
    {"nom_service": "Médecine interne", "categorie": "Médecine", "capacite_lits": 30},
    {"nom_service": "Orthopédie", "categorie": "Chirurgie", "capacite_lits": 25},
]

TYPES_ADMISSION = ["Urgence", "Programmée", "Transfert"]

print(f"Configuration : {N_PATIENTS} patients / {N_HOSPITALISATIONS} hospitalisations / {len(SERVICES)} services")

Configuration : 1000 patients / 1000 hospitalisations / 9 services


## 2. Génération de la table `services`

In [3]:
df_services = pd.DataFrame(SERVICES)
df_services.insert(0, "id_service", range(1, len(df_services) + 1))

df_services.head(10)

,id_service,nom_service,categorie,capacite_lits
0,1,Cardiologie,Médecine,30
1,2,Pédiatrie,Médecine,25
2,3,Chirurgie générale,Chirurgie,35
3,4,Urgences,Urgences,20
4,5,Maternité,Gynéco-Obstétrique,40
5,6,Oncologie,Médecine,20
6,7,Néphrologie,Médecine,15
7,8,Médecine interne,Médecine,30
8,9,Orthopédie,Chirurgie,25


## 3. Génération de la table `patients`

In [5]:
PRENOMS_MASCULINS_MAROC = [
    "Mohammed", "Ahmed", "Youssef", "Hamza", "Ayoub", "Karim", "Rachid", "Omar",
    "Yassine", "Amine", "Anas", "Ismail", "Khalid", "Reda", "Soufiane", "Adil",
    "Mehdi", "Nabil", "Tarik", "Zakaria", "Hassan", "Abdellah", "Said", "Driss",
    "Abderrahim", "Aziz", "Bilal", "Brahim", "Chakib", "Fouad", "Hamid", "Idriss",
    "Jamal", "Karam", "Lahcen", "Mounir", "Noureddine", "Othmane", "Rida", "Samir",
    "Taha", "Walid", "Yassir", "Ziad", "Abdelkader", "Abdelali", "Achraf", "Badr",
    "Chafik", "Elhassane", "Fahd", "Ghali", "Hicham", "Imad", "Jawad", "Kamal",
    "Marouane", "Nourdine", "Othman", "Rachad"
]

PRENOMS_FEMININS_MAROC = [
    "Fatima", "Khadija", "Salma", "Meryem", "Amina", "Nadia", "Hiba", "Imane",
    "Sara", "Zineb", "Hajar", "Loubna", "Ghita", "Kenza", "Sanae", "Naima",
    "Btissam", "Ilham", "Samira", "Asmaa", "Karima", "Latifa", "Wafaa", "Souad",
    "Aicha", "Bouchra", "Chaimae", "Dounia", "Fadwa", "Hanane", "Ibtissam", "Jihane",
    "Karima", "Laila", "Malak", "Nawal", "Ouafae", "Rania", "Siham", "Touria",
    "Widad", "Yasmine", "Zahra", "Amal", "Basma", "Chahd", "Doha", "Fatiha",
    "Ghizlane", "Houda", "Ikram", "Jamila", "Khaoula", "Leila", "Meryam", "Nisrine",
    "Oumaima", "Rachida", "Safaa", "Yousra"
]

NOMS_FAMILLE_MAROC = [
    "Alaoui", "Bennani", "El Idrissi", "Tazi", "Berrada", "Fassi", "El Amrani",
    "Chraibi", "Benjelloun", "Lahlou", "El Ouazzani", "Sbai", "Cherkaoui",
    "El Fassi", "Bennis", "Kabbaj", "Belhaj", "Ziani", "El Yousfi", "Guessous",
    "Idrissi", "Benkirane", "Ouali", "Sekkat", "Alami", "Benslimane", "Chaoui",
    "Dahbi", "El Fenni", "Ghazi", "Hakimi", "Jabri", "Kadiri", "Loukili", "Mansouri",
    "Naciri", "Ouahbi", "Qadiri", "Rifi", "Sahli", "Tahiri", "Zaki", "Amrani",
    "Bakkali", "Chentouf", "Doukkali", "El Gharbi", "Filali", "Habibi", "Jouahri",
    "Kettani"
]

def generer_patient(id_patient):
    sexe = random.choice(["M", "F"])
    prenom = random.choice(PRENOMS_MASCULINS_MAROC) if sexe == "M" else random.choice(PRENOMS_FEMININS_MAROC)
    date_naissance = fake.date_of_birth(minimum_age=0, maximum_age=95)
    age = (pd.Timestamp.today().date() - date_naissance).days // 365

    return {
        "id_patient": id_patient,
        "nom": random.choice(NOMS_FAMILLE_MAROC),
        "prenom": prenom,
        "date_naissance": date_naissance,
        "age": age,
        "sexe": sexe,
        "ville": random.choice(VILLES_MAROC),
    }

patients_list = [generer_patient(i) for i in range(1, N_PATIENTS + 1)]
df_patients = pd.DataFrame(patients_list)

df_patients.head(10)

,id_patient,nom,prenom,date_naissance,age,sexe,ville
0,1,Jabri,Basma,1941-03-08,85,F,Rabat
1,2,Berrada,Yassine,1998-09-01,27,M,Oujda
2,3,Doukkali,Nourdine,1977-07-31,48,M,Oujda
3,4,Bakkali,Ghali,1937-10-26,88,M,Inezgane
4,5,Ouali,Zahra,1943-10-21,82,F,Taroudant
5,6,Mansouri,Mounir,1989-12-30,36,M,Tanger
6,7,El Ouazzani,Meryam,1962-02-07,64,F,Taroudant
7,8,Sekkat,Fahd,1940-11-08,85,M,Kénitra
8,9,Cherkaoui,Omar,1976-11-11,49,M,Essaouira
9,10,Kettani,Adil,2012-11-15,13,M,Oujda


## 4. Génération de la table `hospitalisations`

Chaque hospitalisation est liée à un patient et un service existants.
La durée de séjour suit une loi exponentielle tronquée (beaucoup de courts séjours, quelques longs séjours) pour rester réaliste.

In [6]:
def generer_hospitalisation(id_hospitalisation):
    id_patient = random.randint(1, N_PATIENTS)
    id_service = random.randint(1, len(df_services))

    # Date d'entrée aléatoire sur les 3 dernières années (pour avoir de la saisonnalité)
    date_entree = fake.date_between(start_date="-3y", end_date="today")

    # Durée de séjour réaliste : moyenne ~6 jours, jamais 0, plafonnée à 60 jours
    duree_sejour = int(np.clip(np.random.exponential(scale=6), 1, 60))
    date_sortie = date_entree + timedelta(days=duree_sejour)

    return {
        "id_hospitalisation": id_hospitalisation,
        "id_patient": id_patient,
        "id_service": id_service,
        "date_entree": date_entree,
        "date_sortie": date_sortie,
        "type_admission": random.choice(TYPES_ADMISSION),
    }

hospitalisations_list = [generer_hospitalisation(i) for i in range(1, N_HOSPITALISATIONS + 1)]
df_hospitalisations = pd.DataFrame(hospitalisations_list)

df_hospitalisations.head(10)

,id_hospitalisation,id_patient,id_service,date_entree,date_sortie,type_admission
0,1,408,2,2025-10-05,2025-10-07,Transfert
1,2,168,1,2025-01-11,2025-01-29,Urgence
2,3,834,5,2025-12-29,2026-01-05,Programmée
3,4,860,2,2024-07-01,2024-07-06,Programmée
4,5,763,6,2024-03-09,2024-03-10,Programmée
5,6,370,6,2023-12-08,2023-12-09,Transfert
6,7,571,5,2024-07-25,2024-07-26,Programmée
7,8,21,7,2026-04-21,2026-05-03,Urgence
8,9,360,4,2024-07-14,2024-07-19,Transfert
9,10,590,9,2024-08-23,2024-08-30,Urgence


## 5. Injection volontaire d'anomalies

Conformément au cahier des charges, on injecte délibérément :
- des **valeurs manquantes** (dates, service, âge)
- des **doublons** (patients et hospitalisations)
- des **incohérences de dates** (`date_sortie < date_entree`)
- des **âges invalides** (négatifs ou > 120)

Le taux de chaque anomalie est documenté ci-dessous — il servira de référence pour le rapport qualité du notebook suivant.

In [7]:
TAUX_VALEURS_MANQUANTES = 0.04   # 4%
TAUX_DOUBLONS = 0.02             # 2%
TAUX_DATES_INCOHERENTES = 0.015  # 1.5%
TAUX_AGES_INVALIDES = 0.01       # 1%

rapport_anomalies = {}

In [8]:
# --- Valeurs manquantes dans hospitalisations (date_sortie, id_service) ---
n_manquants = int(len(df_hospitalisations) * TAUX_VALEURS_MANQUANTES)
idx_manquants = np.random.choice(df_hospitalisations.index, size=n_manquants, replace=False)

# On répartit entre les deux colonnes
moitie = n_manquants // 2
df_hospitalisations.loc[idx_manquants[:moitie], "date_sortie"] = np.nan
df_hospitalisations.loc[idx_manquants[moitie:], "id_service"] = np.nan

# --- Valeurs manquantes dans patients (âge, ville) ---
n_manquants_patients = int(len(df_patients) * TAUX_VALEURS_MANQUANTES)
idx_manquants_p = np.random.choice(df_patients.index, size=n_manquants_patients, replace=False)
df_patients.loc[idx_manquants_p, "age"] = np.nan

rapport_anomalies["valeurs_manquantes_hospitalisations"] = n_manquants
rapport_anomalies["valeurs_manquantes_patients"] = n_manquants_patients

print(f"{n_manquants} valeurs manquantes injectées dans hospitalisations")
print(f"{n_manquants_patients} valeurs manquantes injectées dans patients")

40 valeurs manquantes injectées dans hospitalisations
40 valeurs manquantes injectées dans patients


In [9]:
# --- Doublons dans hospitalisations ---
n_doublons_hosp = int(len(df_hospitalisations) * TAUX_DOUBLONS)
lignes_a_dupliquer = df_hospitalisations.sample(n=n_doublons_hosp, random_state=SEED)
df_hospitalisations = pd.concat([df_hospitalisations, lignes_a_dupliquer], ignore_index=True)

# --- Doublons dans patients ---
n_doublons_patients = int(len(df_patients) * TAUX_DOUBLONS)
lignes_a_dupliquer_p = df_patients.sample(n=n_doublons_patients, random_state=SEED)
df_patients = pd.concat([df_patients, lignes_a_dupliquer_p], ignore_index=True)

rapport_anomalies["doublons_hospitalisations"] = n_doublons_hosp
rapport_anomalies["doublons_patients"] = n_doublons_patients

print(f"{n_doublons_hosp} doublons injectés dans hospitalisations")
print(f"{n_doublons_patients} doublons injectés dans patients")

20 doublons injectés dans hospitalisations
20 doublons injectés dans patients


In [10]:
# --- Dates incohérentes : date_sortie < date_entree ---
mask_dates_valides = df_hospitalisations["date_sortie"].notna()
indices_valides = df_hospitalisations[mask_dates_valides].index

n_dates_incoherentes = int(len(df_hospitalisations) * TAUX_DATES_INCOHERENTES)
idx_dates_incoherentes = np.random.choice(indices_valides, size=min(n_dates_incoherentes, len(indices_valides)), replace=False)

# On inverse : la sortie devient antérieure à l'entrée
for idx in idx_dates_incoherentes:
    entree = df_hospitalisations.loc[idx, "date_entree"]
    df_hospitalisations.loc[idx, "date_sortie"] = entree - timedelta(days=random.randint(1, 5))

rapport_anomalies["dates_incoherentes"] = len(idx_dates_incoherentes)
print(f"{len(idx_dates_incoherentes)} incohérences de dates injectées")

15 incohérences de dates injectées


In [11]:
# --- Âges invalides (négatifs ou > 120) ---
mask_age_valide = df_patients["age"].notna()
indices_age_valides = df_patients[mask_age_valide].index

n_ages_invalides = int(len(df_patients) * TAUX_AGES_INVALIDES)
idx_ages_invalides = np.random.choice(indices_age_valides, size=min(n_ages_invalides, len(indices_age_valides)), replace=False)

for idx in idx_ages_invalides:
    df_patients.loc[idx, "age"] = random.choice([-5, -1, 150, 200])

rapport_anomalies["ages_invalides"] = len(idx_ages_invalides)
print(f"{len(idx_ages_invalides)} âges invalides injectés")

10 âges invalides injectés


## 6. Résumé des anomalies injectées

In [12]:
df_rapport_anomalies = pd.DataFrame(
    list(rapport_anomalies.items()), columns=["type_anomalie", "nb_lignes_affectees"]
)
df_rapport_anomalies

,type_anomalie,nb_lignes_affectees
0,valeurs_manquantes_hospitalisations,40
1,valeurs_manquantes_patients,40
2,doublons_hospitalisations,20
3,doublons_patients,20
4,dates_incoherentes,15
5,ages_invalides,10


## 7. Export des fichiers CSV bruts

In [13]:
df_patients.to_csv(f"{OUTPUT_DIR}/patients.csv", index=False)
df_services.to_csv(f"{OUTPUT_DIR}/services.csv", index=False)
df_hospitalisations.to_csv(f"{OUTPUT_DIR}/hospitalisations.csv", index=False)
df_rapport_anomalies.to_csv(f"{OUTPUT_DIR}/rapport_anomalies_injectees.csv", index=False)

print("Fichiers exportés dans data/raw/ ✅")
print(f"- patients.csv : {len(df_patients)} lignes")
print(f"- services.csv : {len(df_services)} lignes")
print(f"- hospitalisations.csv : {len(df_hospitalisations)} lignes")

Fichiers exportés dans data/raw/ ✅
- patients.csv : 1020 lignes
- services.csv : 9 lignes
- hospitalisations.csv : 1020 lignes


## 8. Vérification rapide (sanity check)

In [14]:
print("Aperçu patients :")
display(df_patients.info())

print("\nAperçu hospitalisations :")
display(df_hospitalisations.info())

print("\nRépartition par ville :")
display(df_patients["ville"].value_counts())

Aperçu patients :
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_patient      1020 non-null   int64  
 1   nom             1020 non-null   str    
 2   prenom          1020 non-null   str    
 3   date_naissance  1020 non-null   object 
 4   age             980 non-null    float64
 5   sexe            1020 non-null   str    
 6   ville           1020 non-null   str    
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 55.9+ KB


None


Aperçu hospitalisations :
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_hospitalisation  1020 non-null   int64  
 1   id_patient          1020 non-null   int64  
 2   id_service          1000 non-null   float64
 3   date_entree         1020 non-null   object 
 4   date_sortie         999 non-null    object 
 5   type_admission      1020 non-null   str    
dtypes: float64(1), int64(2), object(2), str(1)
memory usage: 47.9+ KB


None


Répartition par ville :


ville
Essaouira           78
Kénitra             76
Marrakech           73
Safi                73
Tétouan             72
Oujda               69
Taroudant           69
Chtouka Aït Baha    69
Meknès              68
Inezgane            66
Rabat               65
Tanger              64
Casablanca          62
Agadir              59
Fès                 57
Name: count, dtype: int64

---
### Prochaine étape
Ouvrir `02_audit_qualite_donnees.ipynb` pour auditer et nettoyer ces fichiers bruts.